# 01 · Instrument nodes

Block 1 check: run each instrument node alone on a hand-made state — preflight, status, move, set FOV, overview, EDS — plus one forced error.

**Before running (twin):** `uv run python scripts/twin.py start` in a terminal.

**Real microscope:** `TARGET = "real"`, real stage coordinates in `SETTINGS`, `STEM_REAL_MCP_URL` in `.env`.

Each node returns only what it changed; `apply` merges it into a **new** state dict, so a saved `before` state stays as it was (needed for undo).

In [ ]:
TARGET = "twin"   # "twin" or "real"

SETTINGS = {
    "target": TARGET,
    "target_xy_m": (5e-9, 0.0),     # move node: stage x, y (twin repeats every FOV: avoid multiples of it)
    "tuning_fov_m": 40e-9,          # set_fov node (the twin fails at 50 / 100 / 200 nm)
    "run_tag": "block1",
}

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu

from stem_agent.config import TARGETS
from stem_agent.data import tiled_io
from stem_agent.mcp import session
from stem_agent.nodes import common
from stem_agent.nodes.eds_node import eds_node
from stem_agent.nodes.move_stage_node import move_stage_node, undo_move
from stem_agent.nodes.overview_node import overview_node
from stem_agent.nodes.preflight_node import preflight_node
from stem_agent.nodes.set_fov_node import set_fov_node, undo_fov
from stem_agent.nodes.status_node import status_end_node, status_start_node


def apply(state, update):
    """Merge a node's update into a new state dict."""
    print("->", sorted(update), "| run_status:", update.get("run_status", state.get("run_status")))
    if "errors" in update:
        print("ERROR:", update["errors"])
    return {**state, **update}


def show(images, titles, points=None):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5), squeeze=False)
    for ax, image, title in zip(axes[0], images, titles):
        ax.imshow(image, cmap="gray")
        for i, (row, col) in enumerate(points or [], start=1):
            ax.plot(col, row, "o", mfc="none", mec="tab:red", ms=14, mew=2)
            ax.text(col + 10, row - 10, str(i), color="tab:red", weight="bold")
        ax.set_title(title)
        ax.axis("off")
    plt.show()


state = {"settings": SETTINGS}

## 1 · Preflight
Connects, checks all 14 tools, creates the run folder. Expected: `run_status: running`.

In [ ]:
state = apply(state, await preflight_node(state))
state["run"]["run_dir"]

## 2 · Status start
Expected on the twin: `ready: True`, all devices `ON`, warnings only for fields the twin does not report.

In [ ]:
state = apply(state, await status_start_node(state))
snap = state["start"]
{key: snap[key] for key in ("stage", "fov_m", "stem_mode", "devices", "ready", "problems", "warnings")}

## 3 · Move
Move to `target_xy_m`, image before and after, then undo. Expected: readback within 10 nm, image shifted, back at the start.

In [ ]:
_, image_before = await common.acquire_haadf()
before = state
state = apply(state, await move_stage_node(state))
_, image_after = await common.acquire_haadf()
show([image_before, image_after], ["before move", f"after move to {SETTINGS['target_xy_m']} m"])

await undo_move(before)
state = apply(state, {"current": {**state["current"], "stage": await common.get_stage()}})
print("stage after undo:", state["current"]["stage"])

## 4 · Set FOV
Zoom to `tuning_fov_m`, image, then undo. Expected: readback within 1 %, wider view, back to the start FOV.

The twin rebuilds its sample when the FOV changes, so the scene may differ between the two images.

In [ ]:
before = state
state = apply(state, await set_fov_node(state))
_, image_zoom = await common.acquire_haadf()

await undo_fov(before)
state = apply(state, {"current": {**state["current"], "fov_m": await session.call("get_fov")}})
_, image_back = await common.acquire_haadf()
show([image_zoom, image_back],
     [f"FOV {SETTINGS['tuning_fov_m'] * 1e9:.0f} nm", f"FOV back to {state['current']['fov_m'] * 1e9:.0f} nm"])

## 5 · Overview
Expected: Tiled key, image shape, `overview.png` in the run folder.

In [ ]:
state = apply(state, await overview_node(state))
image = tiled_io.read_image(state["overview"]["tiled_key"])
state["overview"]

## 6 · EDS and beam order
Test-only point picking (real segmentation comes in block 3): the 3 widest bright particles (deepest interior pixel) and the darkest background point.

Then the EDS node runs with **both** beam orders (twin: `scripts/twin.py` sets the scan to 128 px, the size of the twin's particle map, otherwise its EDS misses the particles). On the twin a particle gives its composition (max fraction ≳ 0.3) and background gives noise (≲ 0.15). The order where the 3 bright points hit and the dark point misses is the right one for this target.

In [ ]:
smooth = ndi.gaussian_filter(image, 1.5 * image.shape[0] / 128)   # same blur at 128 or 512 px
mask = smooth > threshold_otsu(smooth)
labels, n = ndi.label(mask)
depth = ndi.distance_transform_edt(mask)
widest = np.argsort(ndi.maximum(depth, labels, range(1, n + 1)))[::-1][:3] + 1
bright = [ndi.maximum_position(depth, labels, label) for label in widest]
dark = np.unravel_index(np.argmin(smooth), smooth.shape)

points = bright + [dark]
particles = [{"id": i, "row": float(r), "col": float(c)} for i, (r, c) in enumerate(points, start=1)]
show([image], ["EDS points: 1-3 particles, 4 background"], points)
particles

In [ ]:
HIT = 0.25   # twin: max element fraction on a particle vs background noise

score = {}
for order in ("col_row", "row_col"):
    trial = {**state, "particles": particles, "settings": {**SETTINGS, "beam_order": order}}
    update = await eds_node(trial)
    if "errors" in update:
        print(order, "ERROR:", update["errors"])
        continue
    hits = [bool(np.load(r["npy_path"]).max() > HIT) for r in update["spectra"]]
    score[order] = sum(hits[:3]) + (not hits[3])
    print(f"{order}: particles {['hit' if h else 'miss' for h in hits[:3]]}, "
          f"background {'hit' if hits[3] else 'miss'} -> score {score[order]}/4\n")

best = max(score, key=score.get)
print(f"best beam order for {TARGET}: {best!r}; config has {TARGETS[TARGET]['beam_order']!r}")

EDS with the configured order (no override), stored in the state:

In [ ]:
state = apply({**state, "particles": particles}, await eds_node({**state, "particles": particles}))
Image(Path(state["run"]["run_dir"]) / "eds.png")

## 7 · Forced error
A 1 mm move is over the 100 µm limit. Expected: `run_status: failed`, an error message, the stage unchanged.

In [ ]:
stage_before = await common.get_stage()
bad = {**state, "settings": {**SETTINGS, "target_xy_m": (1e-3, 0.0)}}
update = await move_stage_node(bad)
print(update)
print("stage unchanged:", await common.get_stage() == stage_before)

## 8 · Status end
Final snapshot, then the files this run produced.

In [ ]:
state = apply(state, await status_end_node(state))
run_dir = Path(state["run"]["run_dir"])
sorted(str(p.relative_to(run_dir)) for p in run_dir.rglob("*") if p.is_file())